# Clip Embedding — SigLIP (Kaggle offline)

Decode frame trực tiếp từ video gốc tại timestamp của `ClipWindow`, encode bằng đúng SigLIP revision của Frame Embedding, rồi masked-mean thành clip vector. Chỉ đọc Kaggle input và chỉ ghi `/kaggle/working`; không dùng `clip_path`, PostgreSQL, R2, upload, caption hay ASR.


In [ ]:
# Configuration
from pathlib import Path
INPUT_DIR = Path("/kaggle/input/btc-clip-input")
OUTPUT_DIR = Path("/kaggle/working/clip_embedding_output")
VIDEOS_FILE = INPUT_DIR / "videos.csv"
SHOT_FILE = INPUT_DIR / "shot.csv"            # accepts shots.csv fallback
CLIP_FILE = INPUT_DIR / "clipwindow.csv"
VIDEO_CACHE_DIR = OUTPUT_DIR / "video_cache"
VIDEO_START = 0; VIDEO_END = None
FRAMES_PER_CLIP = 16
MAX_CLIPS_PER_DECODE_UNIT = 16
IMAGE_BATCH_SIZE = 256
MIN_BATCH_SIZE = 8
AUTO_TUNE_BATCH = True
AUTO_TUNE_BATCH_CANDIDATES = (64, 128, 192, 256, 384, 512)
DECODE_WORKERS = 1  # one bounded producer overlaps grouped PyAV decode with GPU inference
DECODE_TOLERANCE_MS = 500
DECODE_GROUP_GAP_MS = 2_000
ROWS_PER_SHARD = 25_000
INDEX_VERSION = 1
RUN_ID = None
ALLOW_CHECKPOINT_OVERRIDE = False
DOWNLOAD_RETRIES = 3
DOWNLOAD_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; ClipEmbedding/1.0)", "Accept": "video/mp4,video/*,*/*;q=0.8"}

MODEL_ID = "google/siglip2-base-patch16-224"
MODEL_REVISION = "75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2"
MODEL_NAME = "siglip2-base-patch16-224"
MODEL_BACKEND = "transformers"
assert FRAMES_PER_CLIP == 16 and 0 < MIN_BATCH_SIZE <= IMAGE_BATCH_SIZE


In [ ]:
# Preflight. Kaggle image must provide torch, transformers, PyAV, faiss and pyarrow.
from __future__ import annotations
import concurrent.futures, csv, gc, hashlib, json, math, os, shutil, subprocess, time, traceback, urllib.request
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import av, faiss, numpy as np, pyarrow as pa, pyarrow.parquet as pq, torch
from PIL import Image
from transformers import AutoModel, AutoProcessor
if not torch.cuda.is_available(): raise RuntimeError("CUDA is mandatory for this notebook; select a Kaggle GPU accelerator.")
if shutil.which("ffprobe") is None: raise RuntimeError("ffprobe is required to validate downloaded videos.")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True; torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda"); AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print({"gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda, "torch": torch.__version__, "compute_dtype": str(AMP_DTYPE)})


In [ ]:
def discover_clip_input() -> None:
    global INPUT_DIR, VIDEOS_FILE, SHOT_FILE, CLIP_FILE, VIDEO_CACHE_DIR
    preferred = [CLIP_FILE] if CLIP_FILE.is_file() else []
    roots = [INPUT_DIR, Path("/kaggle/input")]
    candidates = preferred + sorted({candidate for root in roots if root.is_dir() for candidate in root.rglob("clipwindow.csv")})
    for clip_path in candidates:
        root = clip_path.parent
        videos = root / "videos.csv"
        shot = root / "shot.csv" if (root / "shot.csv").is_file() else root / "shots.csv"
        if videos.is_file() and shot.is_file():
            INPUT_DIR, VIDEOS_FILE, SHOT_FILE, CLIP_FILE = root, videos, shot, clip_path
            VIDEO_CACHE_DIR = OUTPUT_DIR / "video_cache"
            print("Using Clip input:", root)
            return
    raise FileNotFoundError("Drop videos.csv, shot.csv/shots.csv and clipwindow.csv into one input directory")

discover_clip_input()

# Input validation and deterministic planning. Invalid data stops before inference.
REQ_SHOT = {"shot_id","video_id","shot_index","start_ms","end_ms","start_frame_idx","end_frame_idx"}
REQ_CLIP = {"clip_id","shot_id","start_ms","end_ms","start_frame_idx","end_frame_idx","sampling_fps","clip_path"}
def read_csv(path: Path) -> list[dict[str,str]]:
    with path.open(encoding="utf-8-sig", newline="") as f: return list(csv.DictReader(f))
def as_int(v: Any, name: str) -> int:
    value = int(v)
    if str(value) != str(v).strip(): raise ValueError(f"{name} must be an integer")
    return value
def load_inputs() -> tuple[list[dict[str,Any]], dict[str,dict[str,Any]]]:
    shot_path = SHOT_FILE if SHOT_FILE.exists() else INPUT_DIR / "shots.csv"
    if not shot_path.exists() or not CLIP_FILE.exists() or not VIDEOS_FILE.exists(): raise FileNotFoundError("Need videos.csv, shot.csv (or shots.csv), and clipwindow.csv")
    shots, clips, videos = read_csv(shot_path), read_csv(CLIP_FILE), read_csv(VIDEOS_FILE)
    if not shots or not clips or not videos: raise ValueError("Input CSVs may not be empty")
    if set(shots[0]) < REQ_SHOT: raise ValueError(f"shot columns missing: {REQ_SHOT-set(shots[0])}")
    if set(clips[0]) != REQ_CLIP: raise ValueError(f"clipwindow must have exactly 8 columns: got {set(clips[0])}")
    if not ({"video_id", "video_url"} <= set(videos[0]) or {"video_id", "video_path"} <= set(videos[0])): raise ValueError("videos.csv needs video_id and video_url or video_path")
    shot_by_id = {}; video_by_id = {}
    for r in videos:
        vid = r["video_id"].strip()
        if not vid or vid in video_by_id: raise ValueError(f"invalid/duplicate video_id {vid!r}")
        video_by_id[vid] = r
    for r in shots:
        sid = r["shot_id"].strip(); r.update({k: as_int(r[k], k) for k in ("shot_index","start_ms","end_ms","start_frame_idx","end_frame_idx")})
        if not sid or sid in shot_by_id or r["video_id"] not in video_by_id: raise ValueError(f"invalid shot FK/ID {sid!r}")
        if min(r["start_ms"],r["start_frame_idx"]) < 0 or r["end_ms"] <= r["start_ms"] or r["end_frame_idx"] <= r["start_frame_idx"]: raise ValueError(f"bad shot bounds {sid}")
        shot_by_id[sid] = r
    planned=[]; seen=set()
    for r in clips:
        cid, sid = r["clip_id"].strip(), r["shot_id"].strip(); shot = shot_by_id.get(sid)
        if not cid or cid in seen or shot is None: raise ValueError(f"invalid clip FK/ID {cid!r}")
        seen.add(cid); r.update({k: as_int(r[k],k) for k in ("start_ms","end_ms","start_frame_idx","end_frame_idx")}); r["sampling_fps"] = float(r["sampling_fps"])
        if (min(r["start_ms"],r["start_frame_idx"],r["sampling_fps"]) < 0 or r["end_ms"] <= r["start_ms"] or r["end_frame_idx"] <= r["start_frame_idx"] or r["start_ms"] < shot["start_ms"] or r["end_ms"] > shot["end_ms"]): raise ValueError(f"bad clip bounds {cid}")
        r["video_id"] = shot["video_id"]; r["shot_index"] = shot["shot_index"]; planned.append(r)
    return sorted(planned,key=lambda x:(x["video_id"],x["shot_index"],x["start_ms"],x["end_ms"],x["clip_id"])), video_by_id
def requested_times(clip: dict[str,Any]) -> list[int]:
    a,b=clip["start_ms"],clip["end_ms"]; return [min(b-1,a+int(((i+.5)*(b-a))/FRAMES_PER_CLIP)) for i in range(FRAMES_PER_CLIP)]


In [ ]:
# Video resolution plus PyAV nearest-frame decoding. HTTP videos are fetched only on demand.
def sha256_file(path: Path) -> str:
    h=hashlib.sha256();
    with path.open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
def ffprobe_ok(path: Path) -> bool:
    p=subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","default=nw=1:nk=1",str(path)],capture_output=True,text=True)
    try: return p.returncode == 0 and float(p.stdout.strip()) > 0
    except ValueError: return False
def resolve_video(row: dict[str,str], video_id: str) -> tuple[Path,bool]:
    local=row.get("video_path","").strip()
    if local:
        source=Path(local); candidates=[source] if source.is_absolute() else [VIDEOS_FILE.parent/source, INPUT_DIR/source]
        resolved=next((candidate.resolve() for candidate in candidates if candidate.is_file()),None)
        if resolved is not None: return resolved, False
    url=row.get("video_url","").strip()
    if not url.startswith(("http://","https://")): raise RuntimeError("no readable local video_path or HTTP(S) video_url")
    VIDEO_CACHE_DIR.mkdir(parents=True,exist_ok=True); target=VIDEO_CACHE_DIR/f"{video_id}.mp4"
    if target.exists() and ffprobe_ok(target): return target, True
    target.unlink(missing_ok=True)
    for attempt in range(DOWNLOAD_RETRIES):
        try:
            request=urllib.request.Request(url,headers=DOWNLOAD_HEADERS)
            with urllib.request.urlopen(request,timeout=120) as src,target.open("wb") as dst: shutil.copyfileobj(src,dst)
            if ffprobe_ok(target): return target, True
        except Exception:
            target.unlink(missing_ok=True)
            if attempt == DOWNLOAD_RETRIES - 1: raise
            time.sleep(2**attempt)
    raise RuntimeError("downloaded video failed ffprobe validation")
def group_timestamps(requested: list[int]) -> list[list[int]]:
    groups: list[list[int]] = []
    for timestamp in sorted(set(requested)):
        if not groups or timestamp - groups[-1][0] > DECODE_GROUP_GAP_MS:
            groups.append([timestamp])
        else:
            groups[-1].append(timestamp)
    return groups

def decode_requests(path: Path, requested: list[int]) -> dict[int, tuple[int, Image.Image] | None]:
    """Seek once per nearby timestamp group instead of once per requested frame."""
    output: dict[int, tuple[int, Image.Image] | None] = {}
    with av.open(str(path)) as container:
        stream = container.streams.video[0]; time_base = float(stream.time_base)
        for group in group_timestamps(requested):
            start_ms = max(0, group[0] - DECODE_TOLERANCE_MS)
            end_ms = group[-1] + DECODE_TOLERANCE_MS
            container.seek(max(0, int((start_ms / 1000) / time_base)), stream=stream, any_frame=False, backward=True)
            candidates: list[tuple[int, Image.Image]] = []
            for frame in container.decode(stream):
                if frame.pts is None:
                    continue
                actual_ms = round(float(frame.pts * stream.time_base) * 1000)
                if actual_ms > end_ms:
                    break
                if actual_ms >= start_ms:
                    candidates.append((actual_ms, frame.to_image().convert("RGB")))
            for timestamp in group:
                output[timestamp] = min(candidates, key=lambda item: abs(item[0] - timestamp)) if candidates else None
    return output

def decode_unit(path: Path, unit: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, list[int]], dict[int, tuple[int, Image.Image] | None], float]:
    requested = {clip["clip_id"]: requested_times(clip) for clip in unit}
    flat = [timestamp for timestamps in requested.values() for timestamp in timestamps]
    started = time.perf_counter(); decoded = decode_requests(path, flat)
    return unit, requested, decoded, time.perf_counter() - started


In [ ]:
# SigLIP inference. Features, never logits; each image vector is float32 L2 normalized.
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION).to(DEVICE).eval()
BATCH_TUNING_TRIALS: list[dict[str, float | int]] = []

def image_features_exact(images: list[Image.Image]) -> np.ndarray:
    batch = processor(images=images, return_tensors="pt")
    pixels = batch["pixel_values"].pin_memory().to(DEVICE, non_blocking=True)
    with torch.inference_mode(), torch.autocast("cuda", dtype=AMP_DTYPE):
        image_output = model.get_image_features(pixel_values=pixels)
    features = image_output.pooler_output
    if features is None or not isinstance(features, torch.Tensor) or features.ndim != 2:
        raise RuntimeError("SigLIP2 did not return a 2D image pooler_output tensor")
    features = torch.nn.functional.normalize(features.float(), p=2, dim=1)
    return np.ascontiguousarray(features.cpu().numpy(), dtype=np.float32)

def autotune_image_batch(images: list[Image.Image]) -> int:
    if not AUTO_TUNE_BATCH or not images:
        return IMAGE_BATCH_SIZE
    best_size, best_rate = MIN_BATCH_SIZE, 0.0; BATCH_TUNING_TRIALS.clear()
    for candidate in AUTO_TUNE_BATCH_CANDIDATES:
        probe = images[:min(candidate, len(images))]
        if len(probe) < MIN_BATCH_SIZE: continue
        try:
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); started = time.perf_counter()
            image_features_exact(probe)
            torch.cuda.synchronize(); elapsed = max(time.perf_counter() - started, 1e-6); rate = len(probe) / elapsed
            BATCH_TUNING_TRIALS.append({"batch_size": candidate, "probe_images": len(probe), "images_per_second": rate, "peak_vram_bytes": int(torch.cuda.max_memory_allocated())})
            if rate >= best_rate * 0.98: best_size, best_rate = candidate, rate
            if len(probe) < candidate: break
        except (torch.OutOfMemoryError, RuntimeError) as exc:
            if "out of memory" not in str(exc).lower(): raise
            torch.cuda.empty_cache(); break
    print("Batch autotune:", BATCH_TUNING_TRIALS, "selected=", best_size)
    return max(MIN_BATCH_SIZE, best_size)

def image_features(images: list[Image.Image], batch_size: int) -> tuple[np.ndarray, int]:
    chunks=[]; pos=0; current_size=batch_size
    while pos < len(images):
        current=min(current_size,len(images)-pos)
        try:
            chunks.append(image_features_exact(images[pos:pos+current])); pos += current
        except (torch.OutOfMemoryError, RuntimeError) as exc:
            if "out of memory" not in str(exc).lower() or current <= MIN_BATCH_SIZE: raise
            current_size=max(MIN_BATCH_SIZE,current//2); torch.cuda.empty_cache()
    return np.ascontiguousarray(np.concatenate(chunks)), current_size


In [ ]:
# Resumable output writer: a video is marked complete only after its rows are flushed atomically.
META_COLUMNS=["faiss_id","clip_id","shot_id","video_id","start_ms","end_ms","sampled_timestamps_ms","actual_timestamps_ms","valid_sample_count","model_id","model_revision","dimension","normalized","vector_shard","vector_row"]
def atomic_json(path:Path,data:Any)->None:
    tmp=path.with_suffix(path.suffix+".tmp");tmp.write_text(json.dumps(data,indent=2,sort_keys=True),encoding="utf-8");tmp.replace(path)
def config_hash() -> str:
    data={k:globals()[k] for k in ("MODEL_ID","MODEL_REVISION","FRAMES_PER_CLIP","DECODE_TOLERANCE_MS","DECODE_GROUP_GAP_MS","INDEX_VERSION","ROWS_PER_SHARD")}
    return hashlib.sha256(json.dumps(data,sort_keys=True).encode()).hexdigest()
def input_hash() -> str:
    h=hashlib.sha256();
    for p in (VIDEOS_FILE, SHOT_FILE if SHOT_FILE.exists() else INPUT_DIR/"shots.csv", CLIP_FILE): h.update(p.name.encode()); h.update(p.read_bytes())
    return h.hexdigest()
def write_shard(root:Path,shard:int,vectors:list[np.ndarray],meta:list[dict[str,Any]]) -> None:
    vdir=root/"vectors";mdir=root/"metadata";vdir.mkdir(exist_ok=True);mdir.mkdir(exist_ok=True)
    vp=vdir/f"part-{shard:05d}.npy"; mp=mdir/f"part-{shard:05d}.parquet"
    tmp=vp.with_suffix(".tmp.npy"); np.save(tmp,np.ascontiguousarray(np.stack(vectors).astype(np.float32))); tmp.replace(vp)
    pq.write_table(pa.Table.from_pylist(meta),mp.with_suffix(".tmp.parquet"));mp.with_suffix(".tmp.parquet").replace(mp)


In [ ]:
# Main run. A bounded producer decodes the next unit while the GPU encodes the current unit.
def append_failure(path: Path, row: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

def run() -> Path:
    clips, videos = load_inputs(); selected = sorted({item["video_id"] for item in clips})[VIDEO_START:VIDEO_END]
    run_id = RUN_ID or time.strftime("run_%Y%m%d_%H%M%S"); root = OUTPUT_DIR / run_id; root.mkdir(parents=True, exist_ok=True)
    checkpoint = root / "checkpoint.json"
    state = {"input_hash": input_hash(), "config_hash": config_hash(), "completed_video_ids": [], "next_faiss_id": 1, "shards": [], "metrics": {"requested_n": 0, "decoded_n": 0, "success": 0, "failed": 0, "decode_s": 0.0, "encode_s": 0.0, "peak": 0}}
    if checkpoint.exists():
        state = json.loads(checkpoint.read_text())
        if (state["input_hash"] != input_hash() or state["config_hash"] != config_hash()) and not ALLOW_CHECKPOINT_OVERRIDE:
            raise RuntimeError("checkpoint input/config hash mismatch; set explicit override only after review")
    metrics = state.setdefault("metrics", {}); failures = root / "failures.jsonl"
    requested_n=int(metrics.get("requested_n",0)); decoded_n=int(metrics.get("decoded_n",0)); success=int(metrics.get("success",0)); failed=int(metrics.get("failed",0)); decode_s=float(metrics.get("decode_s",0)); encode_s=float(metrics.get("encode_s",0)); peak=int(metrics.get("peak",0))
    all_vec: list[np.ndarray] = []; all_meta: list[dict[str, Any]] = []; shard=len(state["shards"]); final_batch: int | None = None
    for video_id in selected:
        if video_id in state["completed_video_ids"]: continue
        group=[clip for clip in clips if clip["video_id"] == video_id]; video=None; cached=False
        try:
            video,cached=resolve_video(videos[video_id],video_id)
            units=[group[offset:offset+MAX_CLIPS_PER_DECODE_UNIT] for offset in range(0,len(group),MAX_CLIPS_PER_DECODE_UNIT)]
            with concurrent.futures.ThreadPoolExecutor(max_workers=DECODE_WORKERS) as pool:
                future = pool.submit(decode_unit, video, units[0]) if units else None
                for unit_index in range(len(units)):
                    assert future is not None
                    unit, req, decoded, unit_decode_s = future.result(); decode_s += unit_decode_s
                    future = pool.submit(decode_unit, video, units[unit_index+1]) if unit_index+1 < len(units) else None
                    flat=[timestamp for timestamps in req.values() for timestamp in timestamps]; requested_n += len(flat); decoded_n += sum(item is not None for item in decoded.values())
                    valid={timestamp:item for timestamp,item in decoded.items() if item is not None}; images=[item[1] for item in valid.values()]; features={}
                    if images:
                        if final_batch is None: final_batch = autotune_image_batch(images)
                        started=time.perf_counter(); matrix,final_batch=image_features(images,final_batch); encode_s += time.perf_counter()-started; features=dict(zip(valid,matrix))
                    for clip in unit:
                        samples=[]; actual=[]
                        for timestamp in req[clip["clip_id"]]:
                            item=decoded.get(timestamp)
                            if item and clip["start_ms"] <= item[0] < clip["end_ms"] and abs(item[0]-timestamp) <= DECODE_TOLERANCE_MS:
                                samples.append(features[timestamp]); actual.append(item[0])
                        if not samples:
                            append_failure(failures,{"stage":"decode_failed","clip_id":clip["clip_id"],"video_id":video_id}); failed += 1; continue
                        vector=np.mean(np.stack(samples),axis=0).astype(np.float32); vector /= np.linalg.norm(vector)
                        all_vec.append(vector); all_meta.append({"faiss_id":state["next_faiss_id"],"clip_id":clip["clip_id"],"shot_id":clip["shot_id"],"video_id":video_id,"start_ms":clip["start_ms"],"end_ms":clip["end_ms"],"sampled_timestamps_ms":req[clip["clip_id"]],"actual_timestamps_ms":actual,"valid_sample_count":len(samples),"model_id":MODEL_ID,"model_revision":MODEL_REVISION,"dimension":len(vector),"normalized":True,"vector_shard":shard,"vector_row":len(all_vec)-1}); state["next_faiss_id"] += 1; success += 1
                    if len(all_vec) >= ROWS_PER_SHARD:
                        write_shard(root,shard,all_vec,all_meta); state["shards"].append(shard); shard += 1; all_vec=[]; all_meta=[]
                    del decoded,valid,images,features; gc.collect(); peak=max(peak,torch.cuda.max_memory_allocated())
            if all_vec:
                write_shard(root,shard,all_vec,all_meta); state["shards"].append(shard); shard += 1; all_vec=[]; all_meta=[]
            state["completed_video_ids"].append(video_id)
            state["metrics"]={"requested_n":requested_n,"decoded_n":decoded_n,"success":success,"failed":failed,"decode_s":decode_s,"encode_s":encode_s,"peak":peak,"final_batch":final_batch,"batch_tuning_trials":BATCH_TUNING_TRIALS}; atomic_json(checkpoint,state)
            if cached: video.unlink(missing_ok=True)
        except Exception as exc:
            append_failure(failures,{"stage":"video","video_id":video_id,"error":str(exc),"traceback":traceback.format_exc()}); failed += len(group)
            state["metrics"]={"requested_n":requested_n,"decoded_n":decoded_n,"success":success,"failed":failed,"decode_s":decode_s,"encode_s":encode_s,"peak":peak,"final_batch":final_batch,"batch_tuning_trials":BATCH_TUNING_TRIALS}; atomic_json(checkpoint,state)
            if cached and video is not None: video.unlink(missing_ok=True)
    if final_batch is None: final_batch = int(state.get("metrics",{}).get("final_batch") or IMAGE_BATCH_SIZE)
    finalize(root,requested_n,decoded_n,success,failed,decode_s,encode_s,peak,final_batch); return root


In [ ]:
# Deprecated finalizer retained only for notebook-history compatibility. Do not call it.
def _deprecated_finalize(root:Path,requested_n:int,decoded_n:int,success:int,failed:int,decode_s:float,encode_s:float,peak:int,batch:int)->None:
    metas=[]; vecs=[]
    for p in sorted((root/"vectors").glob("part-*.npy")): vecs.append(np.load(p))
    for p in sorted((root/"metadata").glob("part-*.parquet")): metas += pq.read_table(p).to_pylist()
    matrix=np.concatenate(vecs) if vecs else np.empty((0,0),np.float32); dim=matrix.shape[1] if len(matrix) else 0
    if len(matrix)!=len(metas) or not np.isfinite(matrix).all() or (len(matrix) and not np.allclose(np.linalg.norm(matrix,axis=1),1,atol=1e-4)): raise RuntimeError("vector/metadata finite-norm validation failed")
    if len({m["faiss_id"] for m in metas})!=len(metas) or len({m["clip_id"] for m in metas})!=len(metas): raise RuntimeError("duplicate FAISS ID or clip_id")
    index=faiss.IndexIDMap2(faiss.IndexFlatIP(dim));
    if len(matrix): index.add_with_ids(matrix,np.asarray([m["faiss_id"] for m in metas],dtype=np.int64)); D,I=index.search(matrix[:1],1); assert int(I[0,0])==metas[0]["faiss_id"]
    faiss.write_index(index,str(root/"clip.faiss")); assert index.ntotal==len(metas)
    with (root/"clip_embedding_mapping.csv").open("w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=["faiss_id","index_version","clip_id","model_name"]);w.writeheader();w.writerows({"faiss_id":m["faiss_id"],"index_version":INDEX_VERSION,"clip_id":m["clip_id"],"model_name":MODEL_NAME} for m in metas)
    sql="-- Full rebuild artifact only; do not append to an existing production FAISS ID space.\n"+"\n".join(f"INSERT INTO clipembeddingrecord (faiss_id, index_version, clip_id, model_name) VALUES ({m['faiss_id']}, {INDEX_VERSION}, '{m['clip_id'].replace(chr(39),chr(39)*2)}', '{MODEL_NAME}') ON CONFLICT (clip_id, index_version) DO NOTHING;" for m in metas)+"\n";(root/"insert_clip_embedding_records.sql").write_text(sql,encoding="utf-8")
    checks={str(p.relative_to(root)):sha256_file(p) for p in root.rglob("*") if p.is_file() and p.name not in {"manifest.json","summary.json"}}
    manifest={"input_hash":input_hash(),"config_hash":config_hash(),"sampling":"uniform_midpoint_16_v1","pooling":"masked_mean_v1","success_count":len(metas),"failure_count":failed,"dimension":dim,"faiss_checksum":checks.get("clip.faiss"),"checksums":checks};atomic_json(root/"manifest.json",manifest)
    summary={"gpu":torch.cuda.get_device_name(0),"cuda":torch.version.cuda,"model_revision":MODEL_REVISION,"dimension":dim,"compute_dtype":str(AMP_DTYPE),"clips_success":len(metas),"clips_failed":failed,"videos_success":len(json.loads((root/"checkpoint.json").read_text())["completed_video_ids"]),"unique_decoded_frames":decoded_n,"requested_frames":requested_n,"dedup_ratio":(decoded_n/requested_n if requested_n else 1),"decode_seconds":decode_s,"encode_seconds":encode_s,"total_seconds":decode_s+encode_s,"clips_per_second":(len(metas)/(decode_s+encode_s) if decode_s+encode_s else 0),"peak_vram_bytes":peak,"final_batch_size":batch};atomic_json(root/"summary.json",summary)
    print("Download Output:",root)

# Run only after executing the streaming finalizer in the next cell.


In [ ]:
# Streaming finalization: one shard at a time; IndexFlat still owns the final index vectors.
def finalize(root: Path, requested_n: int, decoded_n: int, success: int, failed: int, decode_s: float, encode_s: float, peak: int, batch: int) -> None:
    vector_paths = sorted((root / "vectors").glob("part-*.npy")); metadata_paths = sorted((root / "metadata").glob("part-*.parquet"))
    if len(vector_paths) != len(metadata_paths): raise RuntimeError("vector/metadata shard count mismatch")
    index, dim, total, seen_ids, seen_clips = None, None, 0, set(), set()
    with (root / "clip_embedding_mapping.csv").open("w", newline="", encoding="utf-8") as mapping, (root / "insert_clip_embedding_records.sql").open("w", encoding="utf-8") as sql:
        writer = csv.DictWriter(mapping, fieldnames=["faiss_id", "index_version", "clip_id", "model_name"]); writer.writeheader()
        sql.write("-- Full rebuild artifact only; do not append to an existing production FAISS ID space.\n")
        for vector_path, metadata_path in zip(vector_paths, metadata_paths):
            matrix = np.ascontiguousarray(np.load(vector_path, mmap_mode="r"), dtype=np.float32)
            metadata = pq.read_table(metadata_path).to_pylist()
            if len(matrix) != len(metadata) or matrix.ndim != 2 or not np.isfinite(matrix).all() or not np.allclose(np.linalg.norm(matrix, axis=1), 1, atol=1e-4): raise RuntimeError(f"invalid shard {vector_path.name}")
            if dim is None: dim = matrix.shape[1]; index = faiss.IndexIDMap2(faiss.IndexFlatIP(dim))
            if matrix.shape[1] != dim: raise RuntimeError("embedding dimensions differ across shards")
            expected_shard = int(vector_path.stem.rsplit("-", 1)[1])
            if any(int(row["vector_shard"]) != expected_shard or int(row["vector_row"]) != position for position, row in enumerate(metadata)): raise RuntimeError(f"invalid vector metadata mapping: {metadata_path.name}")
            ids = np.asarray([row["faiss_id"] for row in metadata], dtype=np.int64)
            clips = [row["clip_id"] for row in metadata]
            if len(set(ids)) != len(ids) or len(set(clips)) != len(clips) or seen_ids.intersection(ids) or seen_clips.intersection(clips): raise RuntimeError("duplicate FAISS ID or clip_id")
            index.add_with_ids(matrix, ids); seen_ids.update(map(int, ids)); seen_clips.update(clips); total += len(metadata)
            for row in metadata:
                writer.writerow({"faiss_id": row["faiss_id"], "index_version": INDEX_VERSION, "clip_id": row["clip_id"], "model_name": MODEL_NAME})
                clip_id = str(row["clip_id"]).replace("'", "''")
                sql.write(f"INSERT INTO clipembeddingrecord (faiss_id, index_version, clip_id, model_name) VALUES ({row['faiss_id']}, {INDEX_VERSION}, '{clip_id}', '{MODEL_NAME}') ON CONFLICT (clip_id, index_version) DO NOTHING;\n")
    if index is None: raise RuntimeError("no successful clip embeddings")
    faiss.write_index(index, str(root / "clip.faiss")); assert index.ntotal == total
    failure_count = sum(1 for line in (root / "failures.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()) if (root / "failures.jsonl").is_file() else 0
    total_seconds = decode_s + encode_s
    summary = {"gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda, "compute_dtype": str(AMP_DTYPE), "model_revision": MODEL_REVISION, "dimension": dim, "clips_success": total, "clips_failed": failure_count, "videos_success": len(json.loads((root / "checkpoint.json").read_text(encoding="utf-8"))["completed_video_ids"]), "unique_decoded_frames": decoded_n, "requested_frames": requested_n, "dedup_ratio": (decoded_n / requested_n if requested_n else 1), "decode_seconds": decode_s, "encode_seconds": encode_s, "total_seconds": total_seconds, "clips_per_second": (total / total_seconds if total_seconds else 0), "peak_vram_bytes": peak, "final_batch_size": batch}
    atomic_json(root / "summary.json", summary)
    checkpoint = root / "checkpoint.json"; state = json.loads(checkpoint.read_text(encoding="utf-8")); atomic_json(checkpoint, {**state, "state": "complete"})
    checks = {str(item.relative_to(root)): sha256_file(item) for item in root.rglob("*") if item.is_file() and item.name != "manifest.json"}
    manifest = {"entity_type": "clip", "input_hash": input_hash(), "config_hash": config_hash(), "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "normalized": True, "sampling_version": "uniform_midpoint_16_v1", "pooling_version": "masked_mean_v1", "success_count": total, "failure_count": failure_count, "dimension": dim, "vector_shards": [str(item.relative_to(root)) for item in vector_paths], "metadata_shards": [str(item.relative_to(root)) for item in metadata_paths], "checksums": checks, "faiss_checksum": checks.get("clip.faiss")}
    atomic_json(root / "manifest.json", manifest); print("Download Output:", root)



In [ ]:
# Real run. Execute after reviewing configuration and completing the preceding cells.
artifact_dir = run()
print(f"Completed: {artifact_dir}")


In [ ]:
# Package validated Clip artifacts into one downloadable ZIP.
zip_path = Path(shutil.make_archive(str(artifact_dir), "zip", root_dir=artifact_dir))
required = [path for path in artifact_dir.rglob("*") if path.suffix in {".sql", ".faiss"}]
if not required:
    raise RuntimeError("ZIP packaging refused: validated SQL/FAISS files were not found")
print(f"Download ZIP: {zip_path} ({zip_path.stat().st_size / 1024**2:.1f} MiB)")


## Mandatory test checklist

Before a production run, execute a short mock MP4 test in Kaggle: two overlapping ClipWindow rows must prove midpoint timestamps, timestamp deduplication, normalized vectors, FAISS self-search and unique mapping. A second mock must force out-of-range/tolerance samples and assert a `decode_failed` row without a vector. `finalize()` performs the post-run count/checksum/finite/norm/unique validations.

Cuối run, tải toàn bộ thư mục in ra ở trên. `clip.faiss` và SQL chỉ dành cho một **full rebuild** cùng `INDEX_VERSION`; notebook không chạy SQL hay kết nối database.
